In [1]:
# Python libs
import pandas as pd

# Magics
from helpers import (
    load_sql_magic,
)
load_sql_magic()          # %%sql   — query DataFrames via duckdb (no extra installs)

True

In [2]:
import duckdb

DB_PATH = "./data/ab_events.duckdb"

con = duckdb.connect(DB_PATH, read_only=True)
event_log = con.execute("SELECT * FROM events").df()
con.close()

print(event_log.shape)
event_log.head()

(2024484, 7)


,date,user_id,hash_id,country,experiment,event_type,amount
0,2025-01-01 21:01:12,5,7674613650421074157,US,"{""num01"":""a""}",page_view,NaN
1,2025-01-01 21:02:09,5,7674613650421074157,US,"{""num01"":""a""}",page_view,NaN
2,2025-01-01 14:45:14,6,1310192797669293303,GB,"{""num01"":""a""}",page_view,NaN
3,2025-01-01 14:46:03,6,1310192797669293303,GB,"{""num01"":""a""}",page_view,NaN
4,2025-01-01 22:36:08,7,1750302349509622455,US,"{""num01"":""a""}",page_view,NaN


In [3]:
with duckdb.connect(DB_PATH, read_only=True) as con:
    df = con.execute("SHOW tables").df()

print(df)

                     name
0                  events
1    fct_ab_buckets_daily
2  int_ab_events_bucketed
3           stg_event_log


In [4]:
%%sql

SELECT * FROM event_log


,date,user_id,hash_id,country,experiment,event_type,amount
0,2025-01-01 21:01:12,5,7674613650421074157,US,"{""num01"":""a""}",page_view,NaN
1,2025-01-01 21:02:09,5,7674613650421074157,US,"{""num01"":""a""}",page_view,NaN
2,2025-01-01 14:45:14,6,1310192797669293303,GB,"{""num01"":""a""}",page_view,NaN
3,2025-01-01 14:46:03,6,1310192797669293303,GB,"{""num01"":""a""}",page_view,NaN
4,2025-01-01 22:36:08,7,1750302349509622455,US,"{""num01"":""a""}",page_view,NaN
...,...,...,...,...,...,...,...
2024479,2025-02-28 23:19:03,25000,4860591953457782041,GB,"{""num01"":""a""}",page_view,NaN
2024480,2025-02-28 23:19:44,25000,4860591953457782041,GB,"{""num01"":""a""}",page_view,NaN
2024481,2025-02-28 23:20:04,25000,4860591953457782041,GB,"{""num01"":""a""}",page_view,NaN
2024482,2025-02-28 23:21:08,25000,4860591953457782041,GB,"{""num01"":""a""}",page_view,NaN


In [5]:
with duckdb.connect(DB_PATH, read_only=True) as con:
    fct_ab_buckets_daily = con.execute("SELECT * FROM fct_ab_buckets_daily").df()


In [6]:
%%sql buckets <<

SELECT * FROM fct_ab_buckets_daily


,date_day,country,bucket,experiment_number,experiment_group,total_events,page_view_count,watch_count,add_to_cart_count,purchase_count,total_unique_users,u_page_view,u_watch,u_add_to_cart,u_purchase,purchase_amount
0,2025-01-01,US,172,num01,a,67,55,9,2,1,12,12,8,2,1,26.14
1,2025-01-01,US,161,num01,b,49,42,7,0,0,9,9,7,0,0,0.00
2,2025-01-01,US,154,num01,b,59,52,5,1,1,12,12,4,1,1,141.77
3,2025-01-01,GB,63,num01,a,29,22,6,1,0,8,8,6,1,0,0.00
4,2025-01-01,GB,127,num01,a,15,13,1,0,1,2,2,1,0,1,58.29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69197,2025-02-25,DE,192,num01,a,9,8,1,0,0,1,1,1,0,0,0.00
69198,2025-02-25,DE,121,num01,a,2,2,0,0,0,1,1,0,0,0,0.00
69199,2025-02-25,DE,71,num01,b,6,5,1,0,0,2,2,1,0,0,0.00
69200,2025-02-25,DE,156,num01,a,5,4,0,0,1,2,2,0,0,1,58.34


In [7]:
buckets[
    (buckets.date_day=="2025-01-10")
    & (buckets.country=='US')
]

,date_day,country,bucket,experiment_number,experiment_group,total_events,page_view_count,watch_count,add_to_cart_count,purchase_count,total_unique_users,u_page_view,u_watch,u_add_to_cart,u_purchase,purchase_amount
1009,2025-01-10,US,29,num01,a,23,20,3,0,0,5,5,3,0,0,0.00
1011,2025-01-10,US,125,num01,b,68,58,8,2,0,9,9,6,2,0,0.00
1012,2025-01-10,US,137,num01,a,75,57,11,3,4,16,16,11,3,4,141.29
1013,2025-01-10,US,198,num01,b,82,66,10,4,2,14,14,10,4,2,108.03
1017,2025-01-10,US,179,num01,a,34,26,6,1,1,7,7,6,1,1,41.21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61710,2025-01-10,US,17,num01,b,26,23,3,0,0,4,4,3,0,0,0.00
61711,2025-01-10,US,127,num01,b,38,35,3,0,0,8,8,3,0,0,0.00
61718,2025-01-10,US,91,num01,a,51,42,8,1,0,9,9,7,1,0,0.00
61729,2025-01-10,US,47,num01,a,49,37,9,2,1,9,9,7,2,1,36.09


In [8]:
(
    buckets.groupby('date_day')
    .total_unique_users.sum()
    .to_frame()
    .tail(20)
)
    

,total_unique_users
date_day,
2025-02-09,6580
2025-02-10,6471
2025-02-11,5958
2025-02-12,6242
2025-02-13,6667
2025-02-14,6517
2025-02-15,7371
2025-02-16,7180
2025-02-17,6293


In [9]:
def mde(
    sigma_c,
    sigma_t,
    n_c,
    n_t,
    alpha=0.003,
    beta=0.2,
    df_method="min",
    return_df=False,
):
    """
    Рассчитывает минимально детектируемый абсолютный эффект (MDE)
    для сравнения двух средних.

    Параметры
    ----------
    sigma_c : float
        Стандартное отклонение в контрольной группе (>= 0).
    sigma_t : float
        Стандартное отклонение в тестовой группе (>= 0).
    n_c : int
        Размер контрольной группы (> 1).
    n_t : int
        Размер тестовой группы (> 1).
    alpha : float, default=0.003
        Уровень значимости двустороннего теста.
    beta : float, default=0.2
        Вероятность ошибки второго рода (1 - beta — мощность теста).
    df_method : {"min", "welch", "pooled"}, default="min"
        Способ расчёта числа степеней свободы.
    return_df : bool, default=False
        Если True, дополнительно возвращает число степеней свободы.

    Возвращает
    ----------
    float или tuple
        MDE или (MDE, df), если return_df=True.

    Комментарии
    -----------
    1. Стандартная ошибка разности средних:
    # SE = sqrt{frac{sigma_c^2}{n_c} + frac{sigma_t^2}{n_t}}

    2. Минимально детектируемый эффект:
    # MDE = (t_{1-alpha/2} + t_{1-beta}) cdot SE

    - Квантили берутся из t-распределения.
    - Степени свободы могут считаться разными способами:
      min, pooled или Welch–Satterthwaite.
    - MDE показывает минимальную разницу средних, которую можно обнаружить
      при заданных alpha, beta и размерах выборок.
    """
    if n_c <= 1 or n_t <= 1:
        raise ValueError("n_c и n_t должны быть > 1")

    if sigma_c < 0 or sigma_t < 0:
        raise ValueError("sigma_c и sigma_t должны быть >= 0")

    if not (0 < alpha < 1):
        raise ValueError("alpha должен лежать в интервале (0, 1)")

    if not (0 < beta < 1):
        raise ValueError("beta должен лежать в интервале (0, 1)")

    allowed_methods = {"min", "welch", "pooled"}
    if df_method not in allowed_methods:
        raise ValueError(
            f"df_method должен быть одним из {allowed_methods}, получено: {df_method}"
        )

    se_c_sq = sigma_c**2 / n_c
    se_t_sq = sigma_t**2 / n_t
    se_diff = np.sqrt(se_c_sq + se_t_sq)

    if df_method == "min":
        df = min(n_c, n_t) - 1

    elif df_method == "pooled":
        df = n_c + n_t - 2

    else:  # df_method == "welch"
        numerator = (se_c_sq + se_t_sq) ** 2
        denominator = (se_c_sq**2) / (n_c - 1) + (se_t_sq**2) / (n_t - 1)

        if denominator == 0 or not np.isfinite(numerator) or not np.isfinite(denominator):
            raise ValueError("Не удалось вычислить Welch df: некорректный знаменатель")

        df = numerator / denominator

        if not np.isfinite(df) or df <= 0:
            raise ValueError("Не удалось вычислить Welch df: df не является положительным конечным числом")

    t_alpha = t.ppf(1 - alpha / 2, df=df)
    t_beta = t.ppf(1 - beta, df=df)

    if not np.isfinite(t_alpha) or not np.isfinite(t_beta):
        raise ValueError("Не удалось вычислить квантили t-распределения")

    mde_value = (t_alpha + t_beta) * se_diff

    if return_df:
        return mde_value, df
    return mde_value
    
def get_bucket_metric(
        buckets,
        metric,
        control_group='a',
        treatment_group='b'
):
    """
    Нормализует бакетную метрику и возвращает значения для контрольной
    и тестовой групп.

    Параметры
    ----------
    buckets : pd.DataFrame
        Датафрейм с бакетами и агрегированными метриками.
    metric : str
        Название колонки с метрикой.
    control_group : str, default='a'
        Название контрольной группы в колонке gr.
    treatment_group : str, default='b'
        Название тестовой группы в колонке gr.

    Возвращает
    ----------
    tuple
        Кортеж из трёх объектов:
        - датафрейм с выбранными колонками и нормализованной метрикой;
        - значения metric_sample для control_group;
        - значения metric_sample для treatment_group.

    Комментарии
    -----------
    - Для обычных метрик нормализация идёт по unique_users.
    - Для retention-метрик вида dN_retention нормализация идёт по unique_newusers.
    - Нормализованная метрика считается как отношение значения метрики
      к размеру соответствующей пользовательской базы.
    - Возвращаются отдельно выборки по бакетам для контрольной и тестовой групп.
    """
    res = buckets.copy()

    unique_users_selector = 'unique_users'
    if re.search(r'd[0-9]+_retention', metric):
        unique_users_selector = 'unique_newusers'

    res['metric_sample'] = res[metric] / res[unique_users_selector]
    a = res.query("gr == @control_group").metric_sample
    b = res.query("gr == @treatment_group").metric_sample

    out_columns = ['gr', metric, 'unique_users', 'metric_sample']
    if 'unique_newusers' in res.columns:
        out_columns.insert(3, 'unique_newusers')

    return res[out_columns], a, b

def run(df, test_metrics=None, group_pairs=(('a', 'b'),), alpha=0.003, beta=0.2):
    """
    Выполняет пакетный анализ метрик по бакетам для заданных пар групп.

    Параметры
    ----------
    df : pd.DataFrame
        Датафрейм с бакетными метриками и колонкой группы gr.
    test_metrics : list, default=None
        Список метрик для анализа.
        Если не задан, берутся все колонки справа от unique_users.
    group_pairs : tuple, default=(('a', 'b'),)
        Пары групп вида (control_group, treatment_group).
    alpha : float, default=0.003
        Уровень значимости.
    beta : float, default=0.2
        Вероятность ошибки второго рода.

    Возвращает
    ----------
    pd.DataFrame
        Таблица результатов, где каждая строка содержит статистики для одной
        комбинации метрики и пары групп.

    Комментарии
    -----------
    Для каждой пары групп и метрики функция:
    - получает нормализованные бакетные значения;
    - считает Welch t-test и относительный эффект;
    - оценивает MDE для текущего числа бакетов;
    - сохраняет p-value, t-статистику, эффект в процентах,
      доверительный интервал и сами векторы значений.

    По математике:
    - эффект считается как относительная разница средних между treatment и control;
    - доверительный интервал для эффекта берётся из t_test;
    - monitoring_mde_pct считается как отношение абсолютного MDE
      к baseline_mean в процентах.
    """

    if df.empty:
        raise ValueError("Датафрейм df пустой. Невозможно выполнить анализ.")
    if not (0 < alpha < 1):
        raise ValueError(f"alpha должен быть в диапазоне (0, 1), получено: {alpha}")
    if not (0 < beta < 1):
        raise ValueError(f"beta должен быть в диапазоне (0, 1), получено: {beta}")

    results = []
    if test_metrics is None:
        test_metrics = list(df.loc[:, 'unique_users':].columns)[1:]

    for metric in test_metrics:
        for control_group, treatment_group in group_pairs:
            metric_sample, control, treatment = get_bucket_metric(df, metric=metric, control_group=control_group,
                                                                  treatment_group=treatment_group)
            try:
                (t_stat, p_value,
                 _absolute_stat, _absolute_left_bound, _absolute_right_bound,
                 relative_stat, _relative_pvalue,
                 _relative_t_stat, relative_left_bound, relative_right_bound) = t_test(control, treatment).values()
            except ValueError:
                continue

            metric_control = metric_sample.query("gr == @control_group")[metric].sum().astype("int")
            metric_treatment = metric_sample.query("gr == @treatment_group")[metric].sum().astype("int")

            if metric_control == 0 or metric_treatment == 0:
                continue

            # Суммарное количество наблюдений (уникальных user-day) по всем бакетам для каждой группы
            uniq_users_field = "unique_newusers" if re.search('d[0-9]+_retention', metric) else "unique_users"
            total_obs_control = metric_sample.query("gr == @control_group")[uniq_users_field].sum()
            total_obs_treatment = metric_sample.query("gr == @treatment_group")[uniq_users_field].sum()

            baseline_mean = metric_control / total_obs_control

            # mde и количество наблюдений
            std_c = np.std(control, ddof=1)
            #std_t = np.std(treatment, ddof=1)
            n_c = len(control)
            n_t = len(treatment)
            if n_c == 1 or n_t == 1:
                continue

            implied_mde = mde(sigma_c=std_c, sigma_t=std_c, n_c=n_c, n_t=n_t, alpha=alpha, beta=beta, df_method="min")

            experiment_line = {
                'metric': metric,
                'control': control_group, 'treatment': treatment_group,
                'metric_control': metric_control, 'metric_treatment': metric_treatment,
                'userday_control': total_obs_control,
                'userday_treatment': total_obs_treatment,

                'effect_size_pct': np.round(relative_stat * 100, 4),
                'p_value': p_value,
                'tstat_obs': t_stat,
                'monitoring_mde_pct': implied_mde / baseline_mean * 100,

                'control_vect': control.values,
                'treatment_vect': treatment.values,
                'CI_low_effect': relative_left_bound * 100,
                'CI_high_effect': relative_right_bound * 100,
            }
            results.append(experiment_line)
    return pd.DataFrame(results)

def t_test(x, y, alpha=0.003):
    """
    Сравнивает две независимые выборки с помощью Welch t-test и дополнительно
    оценивает относительный эффект между средними.

    Параметры
    ----------
    x : array-like
        Значения метрики в контрольной группе.
    y : array-like
        Значения метрики в тестовой группе.
    alpha : float, default=0.003
        Уровень значимости для двустороннего доверительного интервала
        относительного эффекта.

    Возвращает
    ----------
    dict
        Словарь с результатами t-теста и оценкой относительного эффекта:
        t_stat, p_value, relative_stat, relative_pvalue, relative_t_stat,
        relative_left_bound, relative_right_bound.

    Комментарии
    -----------
    - Для сравнения средних используется Welch t-test без предположения
      о равенстве дисперсий.
    - Относительный эффект считается как y.mean() / x.mean() - 1.
    - Дисперсия относительного эффекта оценивается дельта-методом.
    - Степени свободы для относительного эффекта считаются по формуле
      Welch–Satterthwaite.
    """
    nx = len(x)
    ny = len(y)
    if nx < 2 or ny < 2:
        raise ValueError("Нужно хотя бы по 2 наблюдения в каждой группе.")

    x_mean = x.mean()
    y_mean = y.mean()
    if np.isclose(x_mean, 0.0):
        raise ValueError("Среднее по control (x_mean) близко к нулю — относительная разница не определена.")
    # относительный эффект
    relative_stat = y_mean / x_mean - 1.0

    # несмещённые оценки дисперсий наблюдений
    sx2 = x.var(ddof=1)
    sy2 = y.var(ddof=1)

    # абсолютный эффект
    absolute_stat = y_mean - x_mean
    absolute_var = sy2 / ny + sx2 / nx
    absolute_se = np.sqrt(absolute_var)
    absolute_df_num = absolute_var ** 2
    absolute_df_den = (
            (sy2 / ny) ** 2 / (ny - 1)
            + (sx2 / nx) ** 2 / (nx - 1)
    )
    if np.isclose(absolute_df_den, 0.0):
        absolute_df = np.nan
        absolute_t_stat = np.nan
        absolute_pvalue = np.nan
        absolute_left_bound = np.nan
        absolute_right_bound = np.nan
    else:
        absolute_df = absolute_df_num / absolute_df_den
        # absolute_t_stat = absolute_stat / absolute_se
        # absolute_pvalue = 2 * t.sf(np.abs(absolute_t_stat), df=absolute_df)
        absolute_q = t.ppf(1 - alpha / 2, df=absolute_df)
        absolute_left_bound = absolute_stat - absolute_q * absolute_se
        absolute_right_bound = absolute_stat + absolute_q * absolute_se

    # Дельта-метод
    a = sy2 / (x_mean ** 2 * ny)
    b = (sx2 * (y_mean ** 2)) / (x_mean ** 4 * nx)
    var_hat = a + b
    se = np.sqrt(var_hat)

    if se == 0:
        raise ValueError("se = 0")
    # t-статистика
    relative_t_stat = relative_stat / se

    # Welch–Satterthwaite df
    denom = (a * a) / (ny - 1) + (b * b) / (nx - 1)
    df = (var_hat ** 2) / denom if denom > 0 else min(nx - 1, ny - 1)

    # Двустороннее p-value и доверительный интервал
    relative_pvalue = 2 * t.sf(np.abs(relative_t_stat), df=df)
    q = t.ppf(1 - alpha / 2, df=df)
    relative_left_bound = relative_stat - q * se
    relative_right_bound = relative_stat + q * se

    t_stat, p_value = stats.ttest_ind(y, x, equal_var=False)

    return {
            "t_stat": t_stat,
            "p_value": p_value,

            "absolute_stat": absolute_stat,
            # "absolute_pvalue": absolute_pvalue,  # то же что и t_stat
            # "absolute_t_stat": absolute_t_stat,  # то же что и p_value
            "absolute_left_bound": absolute_left_bound,
            "absolute_right_bound": absolute_right_bound,

            "relative_stat": relative_stat,
            "relative_pvalue": relative_pvalue,
            "relative_t_stat": relative_t_stat,
            "relative_left_bound": relative_left_bound,
            "relative_right_bound": relative_right_bound
            }



EFFECT_COL = "effect_size_pct"
DROP_COLUMNS_TT = [
    "p_value",
    "pct_of_required_obs", "control_vect", "treatment_vect"]

def _pick_stat_col(df):
    """Выбирает колонку со статистикой (tstat или zstat), если она есть."""
    if "tstat_obs" in df.columns:
        return "tstat_obs"
    if "z_obs" in df.columns:
        return "z_obs"
    return None

def _make_effect_highlighter(*, stat_col, stat_threshold):
    """Создаёт функцию для подсветки эффекта в зависимости от статистики."""
    def highlighter(row):
        c_alpha = None
        if stat_col is not None and pd.notna(row.get(stat_col)):
            try:
                c_alpha = float(row[stat_col])
            except Exception:
                c_alpha = None

        strong = (
                pd.notna(row.get(EFFECT_COL))
                and pd.notna(row.get("monitoring_mde_pct"))
                and abs(row[EFFECT_COL]) >= row["monitoring_mde_pct"]
        )

        style = ""
        if c_alpha is not None:
            if c_alpha <= -stat_threshold:
                style = "background-color: red" if strong else "background-color: #ff9090"
            elif c_alpha >= stat_threshold:
                style = "background-color: green" if strong else "background-color: #90ff90"

        return pd.Series({col: (style if col == EFFECT_COL else "") for col in row.index})

    return highlighter

def _build_thousands_formatters(df, *, threshold=1000, decimals=0):
    """Создаёт форматтеры для числовых колонок с большими значениями."""
    numeric_cols = [
        col
        for col in df.columns
        if pd.api.types.is_numeric_dtype(df[col]) and (df[col].abs() >= threshold).any()
    ]
    return {
        col: (lambda v, _d=decimals, _t=threshold: _format_thousands_with_space(v, decimals=_d, threshold=_t))
        for col in numeric_cols
    }

def _format_thousands_with_space(x, *, decimals=0, threshold=1000):
    """Форматирует число, добавляя пробелы между тысячами при превышении порога."""
    if pd.isna(x):
        return ""
    if isinstance(x, (int, float)) and abs(x) >= threshold:
        fmt = f"{{x:,.{decimals}f}}"
        return fmt.format(x=x).replace(",", " ")
    return x

    
def display_tt(df=None, stat_threshold=3):
    """Отображает DataFrame со стилями и форматированием результатов t/z-теста."""
    if df is None or df.empty:
        print("DataFrame is None or empty")
        return None

    required = {EFFECT_COL, "monitoring_mde_pct"}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"Отсутствуют обязательные колонки: {sorted(missing)}")

    stat_col = _pick_stat_col(df)

    formatted_df = df.copy().drop(columns=DROP_COLUMNS_TT, errors="ignore")

    styler = formatted_df.style.apply(
        _make_effect_highlighter(stat_col=stat_col, stat_threshold=stat_threshold),
        axis=1,
    )

    fmts = _build_thousands_formatters(formatted_df, threshold=1000, decimals=0)

    specific_formatters = {
        "tstat_obs": lambda x: f"{abs(x):.2f}" if pd.notna(x) else "",
        "z_obs": lambda x: f"{abs(x):.2f}" if pd.notna(x) else "",
        EFFECT_COL: lambda x: f"{x:.3f}" if pd.notna(x) else "",
        "monitoring_mde_pct": lambda x: f"{x:.3f}" if pd.notna(x) else "",
        "CI_low_effect": lambda x: f"{x:.3f}" if pd.notna(x) else "",
        "CI_high_effect": lambda x: f"{x:.3f}" if pd.notna(x) else "",
    }
    for col, fmt in specific_formatters.items():
        if col in formatted_df.columns:
            fmts[col] = fmt

    return styler.format(fmts, na_rep="")





# Файл, в который пишет dbt (см. path в profiles.yml). Список таблиц: SHOW ALL TABLES
DB_PATH = "/Users/eugenekomissarov/Documents/localsource/bucky/data/ab_events.duckdb"
MART_TABLE = "fct_ab_buckets_daily"   # <- поставь имя своей mart-модели
EXPERIMENT = "num01"

# Аддитивные метрики из марта; unique_users собирается отдельно из total_unique_users
COUNT_COLUMNS = [
    "total_events", "page_view_count", "watch_count", "add_to_cart_count", "purchase_count",
    "u_page_view", "u_watch", "u_add_to_cart", "u_purchase",
]


def load_buckets(db_path=DB_PATH, table=MART_TABLE, experiment=EXPERIMENT,
                 date_from=None, date_to=None, countries=None) -> pd.DataFrame:
    """Сворачивает март до строк (bucket, gr) с колонками bucket, gr, unique_users, <метрики>."""
    conditions, params = ["experiment_number = ?"], [experiment]
    if date_from:
        conditions.append("date_day >= CAST(? AS DATE)")
        params.append(date_from)
    if date_to:
        conditions.append("date_day <= CAST(? AS DATE)")
        params.append(date_to)
    if countries:
        conditions.append(f"country IN ({', '.join('?' * len(countries))})")
        params.extend(countries)

    metric_sums = ",\n            ".join(f"CAST(SUM({c}) AS BIGINT) AS {c}" for c in COUNT_COLUMNS)
    sql = f"""
        SELECT
            bucket,
            experiment_group AS gr,
            CAST(SUM(total_unique_users) AS BIGINT) AS unique_users,   -- user-days
            {metric_sums},
            SUM(purchase_amount) AS purchase_amount
        FROM {table}
        WHERE {' AND '.join(conditions)}
        GROUP BY bucket, experiment_group
        ORDER BY gr, bucket
    """
    with duckdb.connect(db_path, read_only=True) as con:
        return con.execute(sql, params).df()


buckets = load_buckets(date_from="2025-01-01", date_to="2025-02-28")
print(buckets.groupby("gr").size())   # ожидаем по 200 бакетов на группу
buckets.head()


from itertools import combinations
import re
import numpy as np
from scipy.stats import t
from scipy import stats
import duckdb
import pandas as pd

results = pd.DataFrame()
for i in (0, ):
    tmp = run(buckets[
              (buckets.bucket.notna())
    ], test_metrics=None, group_pairs=list(combinations("".join([i for i in buckets.gr.unique() if len(i)==1]), 2)))
    tmp['dropdown'] = f"all"
    results = pd.concat([results, tmp])

display_tt(results[
        (results.control=='a')
        &(np.abs(results.tstat_obs)>1.96)
        ].sort_values(['metric_control', 'metric', 'effect_size_pct',], ascending=[False, False, False]).reset_index(drop=True))

gr
a    200
b    200
dtype: int64


,metric,control,treatment,metric_control,metric_treatment,userday_control,userday_treatment,effect_size_pct,tstat_obs,monitoring_mde_pct,CI_low_effect,CI_high_effect,dropdown
0,total_events,a,b,1 013 721,1 010 763,196 971,192 857,1.690,3.63,1.723,0.289,3.092,all
1,purchase_amount,a,b,697 835,789 273,196 971,192 857,14.974,7.06,7.558,8.221,21.728,all
2,watch_count,a,b,131 193,138 505,196 971,192 857,7.775,16.51,1.755,6.317,9.234,all
3,u_watch,a,b,117 874,123 515,196 971,192 857,7.017,20.88,1.311,5.977,8.058,all
4,add_to_cart_count,a,b,22 648,24 054,196 971,192 857,8.284,7.39,4.220,4.800,11.768,all
5,u_add_to_cart,a,b,22 141,23 468,196 971,192 857,8.045,7.47,4.090,4.701,11.389,all
6,purchase_count,a,b,12 716,14 387,196 971,192 857,14.868,9.57,5.641,9.910,19.827,all
7,u_purchase,a,b,12 510,14 101,196 971,192 857,14.426,9.58,5.563,9.621,19.231,all


In [8]:
from __future__ import annotations
import re
from functools import partial
from itertools import combinations
from typing import Optional
import duckdb
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import t

# =============================================================================
# Settings
# =============================================================================
DEFAULT_ALPHA, DEFAULT_BETA = 0.003, 0.2
DB_PATH = "/Users/eugenekomissarov/Documents/localsource/bucky/data/ab_events.duckdb"
MART_TABLE, EXPERIMENT = "fct_ab_buckets_daily", "num01"
COUNT_COLUMNS = ["total_events", "page_view_count", "watch_count", "add_to_cart_count", "purchase_count",
                 "u_page_view", "u_watch", "u_add_to_cart", "u_purchase"]
RETENTION_METRIC = re.compile(r"d\d+_retention")
EFFECT_COL = "effect_size_pct"
DROP_COLUMNS_TT = ["p_value", "pct_of_required_obs", "control_vect", "treatment_vect"]

# =============================================================================
# Data
# =============================================================================
def load_buckets(db_path=DB_PATH, table=MART_TABLE, experiment=EXPERIMENT, date_from=None, date_to=None, countries=None):
    """Load and aggregate dbt mart to one row per (bucket, group)."""
    conditions, params = ["experiment_number = ?"], [experiment]
    if date_from:
        conditions.append("date_day >= CAST(? AS DATE)"); params.append(date_from)
    if date_to:
        conditions.append("date_day <= CAST(? AS DATE)"); params.append(date_to)
    if countries:
        conditions.append(f"country IN ({', '.join('?' * len(countries))})"); params.extend(countries)
    metric_sums = ", ".join(f"CAST(SUM({c}) AS BIGINT) AS {c}" for c in COUNT_COLUMNS)
    sql = f"""SELECT bucket, experiment_group AS gr, CAST(SUM(total_unique_users) AS BIGINT) AS unique_users,
              {metric_sums}, SUM(purchase_amount) AS purchase_amount
              FROM {table} WHERE {' AND '.join(conditions)}
              GROUP BY bucket, experiment_group ORDER BY gr, bucket"""
    with duckdb.connect(db_path, read_only=True) as con:
        return con.execute(sql, params).df()

def group_pairs_of(df):
    groups = [g for g in df["gr"].dropna().unique() if len(g) == 1]
    return list(combinations(groups, 2))

# =============================================================================
# Statistics
# =============================================================================
def _check_probability(name, value):
    if not 0 < value < 1:
        raise ValueError(f"{name} must be in (0, 1), got {value}")

def _degrees_of_freedom(method, n_c, n_t, var_c, var_t):
    if method == "min": return min(n_c, n_t) - 1
    if method == "pooled": return n_c + n_t - 2
    numerator = (var_c + var_t) ** 2
    denominator = var_c ** 2 / (n_c - 1) + var_t ** 2 / (n_t - 1)
    if denominator <= 0 or not np.isfinite(numerator) or not np.isfinite(denominator):
        raise ValueError("Cannot calculate Welch degrees of freedom")
    return numerator / denominator

def mde(sigma_c, sigma_t, n_c, n_t, alpha=DEFAULT_ALPHA, beta=DEFAULT_BETA, df_method="min"):
    """Absolute MDE for comparison of two independent means."""
    if n_c <= 1 or n_t <= 1: raise ValueError("n_c and n_t must be > 1")
    if sigma_c < 0 or sigma_t < 0: raise ValueError("sigma must be >= 0")
    _check_probability("alpha", alpha); _check_probability("beta", beta)
    if df_method not in {"min", "welch", "pooled"}: raise ValueError("df_method must be min, welch or pooled")
    var_c, var_t = sigma_c ** 2 / n_c, sigma_t ** 2 / n_t
    df = _degrees_of_freedom(df_method, n_c, n_t, var_c, var_t)
    return (t.ppf(1 - alpha / 2, df) + t.ppf(1 - beta, df)) * np.sqrt(var_c + var_t)

def _relative_effect(x_mean, y_mean, var_x, var_y, nx, ny, alpha):
    """Relative effect y/x - 1 with delta-method CI."""
    effect = y_mean / x_mean - 1
    a = var_y / (x_mean ** 2 * ny)
    b = var_x * y_mean ** 2 / (x_mean ** 4 * nx)
    variance, se = a + b, np.sqrt(a + b)
    if np.isclose(se, 0): raise ValueError("SE is zero")
    denominator = a ** 2 / (ny - 1) + b ** 2 / (nx - 1)
    df = variance ** 2 / denominator if denominator > 0 else min(nx - 1, ny - 1)
    q = t.ppf(1 - alpha / 2, df)
    return effect, effect - q * se, effect + q * se

def t_test(x, y, alpha=DEFAULT_ALPHA):
    """Welch t-test plus relative effect and delta-method CI."""
    x, y = np.asarray(x), np.asarray(y)
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2: raise ValueError("Need at least 2 observations per group")
    x_mean, y_mean = x.mean(), y.mean()
    if np.isclose(x_mean, 0): raise ValueError("Control mean is zero")
    var_x, var_y = x.var(ddof=1), y.var(ddof=1)
    t_stat, p_value = stats.ttest_ind(y, x, equal_var=False)
    effect, left, right = _relative_effect(x_mean, y_mean, var_x, var_y, nx, ny, alpha)
    return {"t_stat": t_stat, "p_value": p_value, "relative_stat": effect,
            "relative_left_bound": left, "relative_right_bound": right}

# =============================================================================
# Bucket analysis
# =============================================================================
def _users_column(metric):
    return "unique_newusers" if RETENTION_METRIC.search(metric) else "unique_users"

def default_test_metrics(df):
    return list(df.loc[:, "unique_users":].columns)[1:]

def get_bucket_metric(df, metric, control_group="a", treatment_group="b"):
    users_col = _users_column(metric)
    columns = list(dict.fromkeys(["gr", metric, "unique_users", users_col]))
    sample = df[columns].copy()
    sample["metric_sample"] = sample[metric] / sample[users_col]
    control = sample.loc[sample["gr"] == control_group, "metric_sample"]
    treatment = sample.loc[sample["gr"] == treatment_group, "metric_sample"]
    return sample, control, treatment

def _compare_groups(df, metric, control_group, treatment_group, alpha, beta) -> Optional[dict]:
    sample, control, treatment = get_bucket_metric(df, metric, control_group, treatment_group)
    control, treatment = control.replace([np.inf, -np.inf], np.nan).dropna(), treatment.replace([np.inf, -np.inf], np.nan).dropna()
    try:
        effect = t_test(control, treatment, alpha=alpha)
    except ValueError:
        return None
    is_c, is_t = sample["gr"] == control_group, sample["gr"] == treatment_group
    metric_c, metric_t = sample.loc[is_c, metric].sum(), sample.loc[is_t, metric].sum()
    users_col = _users_column(metric)
    users_c, users_t = sample.loc[is_c, users_col].sum(), sample.loc[is_t, users_col].sum()
    if metric_c == 0 or metric_t == 0 or users_c == 0 or users_t == 0: return None
    baseline = metric_c / users_c
    std_c = control.std(ddof=1)
    implied_mde = mde(std_c, std_c, len(control), len(treatment), alpha, beta)
    return {"metric": metric, "control": control_group, "treatment": treatment_group,
            "metric_control": metric_c, "metric_treatment": metric_t,
            "userday_control": users_c, "userday_treatment": users_t,
            EFFECT_COL: np.round(effect["relative_stat"] * 100, 4),
            "p_value": effect["p_value"], "tstat_obs": effect["t_stat"],
            "monitoring_mde_pct": implied_mde / baseline * 100,
            "control_vect": control.values, "treatment_vect": treatment.values,
            "CI_low_effect": effect["relative_left_bound"] * 100,
            "CI_high_effect": effect["relative_right_bound"] * 100}

def run(df, test_metrics=None, group_pairs=None, alpha=DEFAULT_ALPHA, beta=DEFAULT_BETA):
    """Aggregate daily mart to bucket level and run tests for every metric and group pair."""
    if df.empty: raise ValueError("df is empty")
    _check_probability("alpha", alpha); _check_probability("beta", beta)

    # Daily mart -> one row per (bucket, group)
    metric_cols = COUNT_COLUMNS + ["purchase_amount"]
    agg = {col: "sum" for col in metric_cols if col in df.columns}
    agg["total_unique_users"] = "sum"
    df = (df.groupby(["bucket", "experiment_group"], as_index=False).agg(agg)
          .rename(columns={"experiment_group": "gr", "total_unique_users": "unique_users"}))

    test_metrics = [col for col in metric_cols if col in df.columns] if test_metrics is None else test_metrics
    group_pairs = group_pairs_of(df) if group_pairs is None else group_pairs
    rows = [row for metric in test_metrics for c, tr in group_pairs
            if (row := _compare_groups(df, metric, c, tr, alpha, beta)) is not None]
    return pd.DataFrame(rows)

def significant_results(results, control_group="a", min_abs_t=1.96):
    return (results[(results["control"] == control_group) & (results["tstat_obs"].abs() > min_abs_t)]
            .sort_values(["metric_control", "metric", EFFECT_COL], ascending=[False, False, False])
            .reset_index(drop=True))

# =============================================================================
# Display
# =============================================================================
def _make_effect_highlighter(stat_threshold):
    def highlighter(row):
        stat, effect, mde_pct = row.get("tstat_obs", np.nan), row.get(EFFECT_COL), row.get("monitoring_mde_pct")
        strong = pd.notna(effect) and pd.notna(mde_pct) and abs(effect) >= mde_pct
        style = ""
        if pd.notna(stat) and stat <= -stat_threshold: style = "background-color: red" if strong else "background-color: #ff9090"
        elif pd.notna(stat) and stat >= stat_threshold: style = "background-color: green" if strong else "background-color: #90ff90"
        return pd.Series({col: style if col == EFFECT_COL else "" for col in row.index})
    return highlighter

def _format_thousands(x):
    if pd.isna(x): return ""
    return f"{x:,.0f}".replace(",", " ") if isinstance(x, (int, float, np.number)) and abs(x) >= 1000 else x

def _number_formatter(digits, absolute=False):
    return lambda x: "" if pd.isna(x) else f"{abs(x) if absolute else x:.{digits}f}"

def display_tt(df=None, stat_threshold=3):
    if df is None or df.empty:
        print("DataFrame is None or empty"); return None
    formatted = df.copy().drop(columns=DROP_COLUMNS_TT, errors="ignore")
    formatters = {col: _format_thousands for col in formatted.columns
                  if pd.api.types.is_numeric_dtype(formatted[col]) and (formatted[col].abs() >= 1000).any()}
    specific = {"tstat_obs": _number_formatter(2, True), EFFECT_COL: _number_formatter(3),
                "monitoring_mde_pct": _number_formatter(3), "CI_low_effect": _number_formatter(3),
                "CI_high_effect": _number_formatter(3)}
    formatters.update({col: fmt for col, fmt in specific.items() if col in formatted.columns})
    return formatted.style.apply(_make_effect_highlighter(stat_threshold), axis=1).format(formatters, na_rep="")

# =============================================================================
# Run
# =============================================================================
results = run(buckets[buckets["bucket"].notna()])
results["dropdown"] = "all"
display_tt(significant_results(results))

,metric,control,treatment,metric_control,metric_treatment,userday_control,userday_treatment,effect_size_pct,tstat_obs,monitoring_mde_pct,CI_low_effect,CI_high_effect,dropdown
0,total_events,a,b,1 013 721,1 010 763,196 971,192 857,1.690,3.63,1.723,0.289,3.092,all
1,purchase_amount,a,b,697 835,789 274,196 971,192 857,14.974,7.06,7.558,8.221,21.728,all
2,watch_count,a,b,131 193,138 505,196 971,192 857,7.775,16.51,1.755,6.317,9.234,all
3,u_watch,a,b,117 874,123 515,196 971,192 857,7.017,20.88,1.311,5.977,8.058,all
4,add_to_cart_count,a,b,22 648,24 054,196 971,192 857,8.284,7.39,4.220,4.800,11.768,all
5,u_add_to_cart,a,b,22 141,23 468,196 971,192 857,8.045,7.47,4.090,4.701,11.389,all
6,purchase_count,a,b,12 716,14 387,196 971,192 857,14.868,9.57,5.641,9.910,19.827,all
7,u_purchase,a,b,12 510,14 101,196 971,192 857,14.426,9.58,5.563,9.621,19.231,all


In [7]:
from database import load_buckets
from analytics import run, significant_results, display_tt

buckets = load_buckets("num01")
results = run(buckets)
display_tt(significant_results(results))

,metric,control,treatment,metric_control,metric_treatment,userday_control,userday_treatment,effect_size_pct,tstat_obs,monitoring_mde_pct,CI_low_effect,CI_high_effect
0,total_events,a,b,1 013 721,1 010 763,196 971,192 857,1.690,3.63,1.723,0.289,3.092
1,purchase_amount,a,b,697 835,789 274,196 971,192 857,14.974,7.06,7.558,8.221,21.728
2,watch_count,a,b,131 193,138 505,196 971,192 857,7.775,16.51,1.755,6.317,9.234
3,u_watch,a,b,117 874,123 515,196 971,192 857,7.017,20.88,1.311,5.977,8.058
4,add_to_cart_count,a,b,22 648,24 054,196 971,192 857,8.284,7.39,4.220,4.800,11.768
5,u_add_to_cart,a,b,22 141,23 468,196 971,192 857,8.045,7.47,4.090,4.701,11.389
6,purchase_count,a,b,12 716,14 387,196 971,192 857,14.868,9.57,5.641,9.910,19.827
7,u_purchase,a,b,12 510,14 101,196 971,192 857,14.426,9.58,5.563,9.621,19.231


In [8]:
buckets["country"].value_counts().head(10)

country
US    23594
GB    23100
DE    22508
Name: count, dtype: int64

In [9]:
buckets_de = load_buckets("num01", country="DE")
results_de = run(buckets_de)
display_tt(significant_results(results_de))

,metric,control,treatment,metric_control,metric_treatment,userday_control,userday_treatment,effect_size_pct,tstat_obs,monitoring_mde_pct,CI_low_effect,CI_high_effect
0,purchase_amount,a,b,115 088,138 229,37 405,38 724,16.750,3.49,17.536,1.231,32.269
1,watch_count,a,b,23 997,26 671,37 405,38 724,7.274,6.54,4.094,3.840,10.708
2,u_watch,a,b,21 544,23 888,37 405,38 724,7.097,8.25,3.243,4.437,9.756
3,add_to_cart_count,a,b,3 910,4 351,37 405,38 724,7.634,2.73,10.780,-1.038,16.306
4,u_add_to_cart,a,b,3 829,4 243,37 405,38 724,7.206,2.64,10.470,-1.246,15.658
5,purchase_count,a,b,2 203,2 741,37 405,38 724,20.210,5.20,13.799,7.519,32.901
6,u_purchase,a,b,2 165,2 690,37 405,38 724,20.067,5.23,13.852,7.526,32.607
